In [21]:
import SimpleITK as sitk
import xml.etree.ElementTree as ET
import os
import numpy as np

#######################################################################################################

# PATHS USED FOR XML COORDINATE MODIFICATION

xml_file = r'C:\Users\acoats\Desktop\Programs\dataset_registration\U-AllysonRegistration\use\case-56\Case-56_registered\frc_xml\ZUNU_vida-xmlTree.xml'
transform_dir = r"C:\Users\acoats\Desktop\Programs\dataset_registration\U-AllysonRegistration\use\case-56\Case-56_registered\Elastix_Registration_Outputs"
transform_file = r'C:\Users\acoats\Desktop\Programs\dataset_registration\U-AllysonRegistration\use\case-56\Case-56_registered\Elastix_Registration_Outputs\TransformParameters.0.h5'

h5_files = [os.path.join(transform_dir, r"TransformParameters.0.h5"), os.path.join(transform_dir, r"TransformParameters.1.h5"), os.path.join(transform_dir, r"TransformParameters.2.h5")]

#######################################################################################################

def extract_xyz_coordinates(xml_file):
 
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Finds all BP elements in the xml file
    branchpoints = root.findall(".//BP")

    coordinates_list = []

    # Iterates over each BP element
    for bp in branchpoints:
        # Extract XYZ coordinates
        x = int(bp.attrib['x'])
        y = int(bp.attrib['y'])
        z = int(bp.attrib['z'])
        coordinates_list.append((x, y, z))

    # Prints or manipulate the coordinates list as needed
    print(coordinates_list)

    return coordinates_list


def write_xml_with_modified_coordinates(xml_file, transformed_coordinates):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Finds all BP elements within the xml file
    branchpoints = root.findall(".//BP")

    # Iterates over each BP element and update coordinates
    for bp, coord in zip(branchpoints, transformed_coordinates):
        bp.attrib['x'] = str(coord[0])
        bp.attrib['y'] = str(coord[1])
        bp.attrib['z'] = str(coord[2])

    # Writes the modified XML to a new file
    new_xml_file = xml_file.replace(".xml", "_modified.xml")
    tree.write(new_xml_file)
    print("Modified xml file -> " + new_xml_file)

# Acquires the xyz coordinates from the original xml file
xml_coordinates = extract_xyz_coordinates(xml_file)
#xml_coordinates = xml_coordinates.T

# Load the transformation parameters from .h5 files
transform1 = sitk.ReadTransform(h5_files[0]) #rigid transform
transform2 = sitk.ReadTransform(h5_files[1]) #12-resolution bspline
transform3 = sitk.ReadTransform(h5_files[2]) #24-resolution bspline

# Loads multiple transforms that are applied left->right
composite_transform = sitk.CompositeTransform([transform1, transform2, transform3])
#print(composite_transform)

# Apply the transformation to each 3D coordinate
transformed_coordinates_3d = []
for coord in xml_coordinates:
    transformed_coord = composite_transform.TransformPoint(coord)
    transformed_coordinates_3d.append(transformed_coord)

# Convert the list of transformed coordinates to numpy array
transformed_coordinates_3d = np.array(transformed_coordinates_3d)
#print(transformed_coordinates_3d)

# Write a new XML file with modified coordinates
write_xml_with_modified_coordinates(xml_file, transformed_coordinates_3d)


[(311, 112, 1), (320, 218, 203), (277, 236, 255), (424, 241, 298), (459, 225, 290), (440, 256, 324), (468, 220, 272), (472, 213, 295), (463, 261, 361), (447, 281, 318), (245, 233, 243), (472, 226, 254), (477, 191, 256), (485, 193, 312), (525, 201, 293), (480, 249, 387), (467, 265, 368), (462, 329, 341), (455, 294, 307), (242, 233, 244), (233, 224, 207), (472, 235, 227), (475, 183, 252), (490, 188, 243), (495, 182, 334), (503, 161, 328), (480, 247, 421), (498, 233, 409), (481, 266, 382), (468, 300, 405), (474, 302, 306), (461, 310, 296), (221, 247, 231), (229, 229, 185), (466, 239, 197), (483, 265, 181), (494, 241, 245), (467, 171, 230), (476, 154, 261), (493, 182, 341), (509, 169, 352), (520, 149, 337), (500, 143, 334), (479, 252, 431), (484, 244, 433), (500, 229, 407), (511, 217, 425), (511, 272, 379), (506, 265, 417), (472, 315, 426), (461, 307, 413), (216, 203, 259), (181, 252, 224), (215, 264, 212), (224, 229, 180), (232, 230, 178), (249, 251, 314), (454, 244, 176), (471, 231, 169)